# Part D — Clinical Review Figure
This notebook inspects `vitals_10k.csv`, documents data-quality rules, and creates one 2×2 clinical review figure saved as a 300-dpi PNG.

**Data handling rules**
1. Parse the mixed date strings with `pd.to_datetime(..., format="mixed", dayfirst=False)`; unparseable dates become missing and are counted.
2. Normalize spelling/case variants of ward names to four canonical ward names.
3. Treat BMI `999` as an invalid sentinel, and non-positive BMI as invalid; set these to missing rather than plotting them as real measurements.
4. Treat heart rate `<= 0` or `> 220 bpm` as implausible entry values; set them to missing rather than silently clipping them.
5. Keep plausible high length-of-stay values: they are skewed but not automatically data-entry errors. The histogram states its bin width.

These are data-quality rules for this teaching dataset, not clinical decision thresholds. Confirm thresholds with a clinical data owner before applying them to real patient data.

In [ ]:
# In Google Colab, upload vitals_10k.csv when prompted.
from google.colab import files
uploaded = files.upload()
csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if csv_name is None:
    raise ValueError("Please upload vitals_10k.csv")
import pandas as pd
df = pd.read_csv(csv_name)
print("Loaded:", csv_name, "| shape:", df.shape)

In [ ]:
# Required initial inspection (run before cleaning)
df.info()
display(df.describe().T)
display(df.ward.value_counts(dropna=False))
print(df.admit_date.head(20).tolist())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgb

# Copy raw data so the original remains available for audit.
clean = df.copy()

# 1) Mixed date formats: parse and report failures.
clean["admit_date"] = pd.to_datetime(clean["admit_date"], format="mixed", errors="coerce", dayfirst=False)
date_failures = int(clean["admit_date"].isna().sum())

# 2) Standardize ward labels.
ward_map = {
    "general medicine": "General Medicine",
    "gen medicine": "General Medicine",
    "cardiology": "Cardiology",
    "cardio": "Cardiology",
    "day surgery": "Day Surgery",
    "icu": "Intensive Care",
    "intensive care": "Intensive Care",
    "i.c.u": "Intensive Care",
}
clean["ward"] = (clean["ward"].astype("string").str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True).map(ward_map).fillna("Other/Unknown"))

# 3) Remove invalid BMI sentinel and non-positive values.
bmi_bad = (clean["bmi"].eq(999) | clean["bmi"].le(0))
bmi_bad_n = int(bmi_bad.sum())
clean.loc[bmi_bad, "bmi"] = np.nan

# 4) Remove implausible heart-rate entries (explicit rule: <=0 or >220 bpm).
hr_bad = clean["heart_rate"].le(0) | clean["heart_rate"].gt(220)
hr_bad_n = int(hr_bad.sum())
clean.loc[hr_bad, "heart_rate"] = np.nan

print("Cleaning audit")
print(f"Unparseable admit_date values: {date_failures}")
print(f"Invalid BMI values set to NaN (BMI == 999 or <= 0): {bmi_bad_n}")
print(f"Implausible heart rates set to NaN (<= 0 or > 220 bpm): {hr_bad_n}")
print("Canonical ward counts:")
display(clean["ward"].value_counts())

In [ ]:
# Build one 2×2 review figure.
sns.set_theme(style="white", context="notebook")
palette = {
    "General Medicine": "#0072B2",  # blue
    "Cardiology": "#E69F00",        # orange
    "Day Surgery": "#009E73",       # bluish green
    "Intensive Care": "#CC79A7",    # reddish purple
    "Other/Unknown": "#666666",
}
fig, axes = plt.subplots(2, 2, figsize=(15, 11), constrained_layout=True)
ax1, ax2, ax3, ax4 = axes.flat

# Panel 1: Length of stay distribution, explicit bin width.
los = clean["los_days"].dropna()
bin_width = 1.0  # day
bins = np.arange(0, np.ceil(los.quantile(0.99) / bin_width) * bin_width + bin_width, bin_width)
ax1.hist(los.clip(upper=los.quantile(0.99)), bins=bins, color="#0072B2", edgecolor="white")
ax1.set_title("Length of stay is strongly right-skewed")
ax1.set_xlabel("Length of stay (days; values above 99th percentile capped for display)")
ax1.set_ylabel("Patient records (count)")
ax1.text(0.98, 0.95, f"Bin width = {bin_width:g} day\nDisplay cap = 99th percentile",
         transform=ax1.transAxes, ha="right", va="top", fontsize=9,
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85, edgecolor="0.8"))

# Panel 2: BMI by ward; long-form tidy data is used directly.
ward_order = ["General Medicine", "Cardiology", "Day Surgery", "Intensive Care", "Other/Unknown"]
sns.boxplot(data=clean, x="ward", y="bmi", order=ward_order, palette=palette,
            showfliers=False, ax=ax2)
ax2.set_title("BMI distributions overlap across wards")
ax2.set_xlabel("Ward")
ax2.set_ylabel("Body mass index (kg/m²)")
ax2.tick_params(axis="x", rotation=15)

# Panel 3: strongest clinically interpretable pair among primary BP measures.
pair = clean[["systolic", "diastolic"]].dropna()
r = pair.corr().iloc[0, 1]
sns.scatterplot(data=pair, x="systolic", y="diastolic", s=18, alpha=0.35,
                color="#0072B2", edgecolor=None, ax=ax3)
sns.regplot(data=pair, x="systolic", y="diastolic", scatter=False, ci=None,
            color="#D55E00", line_kws={"linewidth": 2}, ax=ax3)
ax3.set_title(f"Systolic and diastolic pressure rise together (r = {r:.2f})")
ax3.set_xlabel("Systolic blood pressure (mmHg)")
ax3.set_ylabel("Diastolic blood pressure (mmHg)")

# Panel 4: annotated correlation heatmap; omit IDs, categorical columns, and derived MAP.
corr_cols = ["age", "systolic", "diastolic", "heart_rate", "spo2",
             "bmi", "chol_mmol_l", "glucose_mmol_l", "los_days", "risk_score"]
corr = clean[corr_cols].corr(numeric_only=True)
sns.heatmap(corr, ax=ax4, cmap="vlag", vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", square=True, linewidths=0.5,
            cbar_kws={"label": "Pearson correlation (r)"})
ax4.set_title("Blood-pressure measures are the clearest correlated pair")
ax4.tick_params(axis="x", rotation=45)
ax4.tick_params(axis="y", rotation=0)

for ax in axes.flat:
    sns.despine(ax=ax)

fig.suptitle("Clinical review of 10,000 vital-sign records", fontsize=16, fontweight="bold")
fig.text(0.01, -0.015,
         "Source: Week 3 teaching dataset, vitals_10k.csv. Invalid BMI and heart-rate entries are excluded; "
         "LOS values above the 99th percentile are capped only in the histogram display.",
         ha="left", va="top", fontsize=9)
out_png = "partD_clinical_review.png"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_png)
files.download(out_png)

## Questions answered by the four panels
1. **Distribution:** Is length of stay normally distributed, or skewed? The histogram shows a long right tail; bin width is explicitly declared.
2. **Ward comparison:** Does BMI vary substantially by ward? Compare medians and spread; invalid sentinel/zero values are excluded.
3. **Scatter:** Are systolic and diastolic blood pressure associated? The figure reports Pearson's *r*.
4. **Correlation:** Which numeric measurements co-vary most strongly? The annotated heatmap makes the correlation magnitude and direction explicit.

**Important:** This is a descriptive review of a synthetic teaching dataset, not a diagnostic tool. Do not interpret correlations as causation.